In [ ]:
# =========================
# USER PARAMETERS
# =========================

resume_path = "../resume_samples.pdf"   # e.g. "data/Shashidhar_Resume.pdf"

skill_dataset_paths = [
    "../data/esco_skills.csv",
    "../data/onet_skills.xlsx",
    "../data/skills.txt",
]

jd_texts = '''Jobs
Services
Search by Skills, Company or Job Title
Please enter a job title/role/skill to search
Location
Please enter a job title/role/skill to search
Experience
Please enter a job title/role/skill to search

9
Shashidhar Satish Hegde
Hi, Shashidhar Satish Hegde
I
AI back-end Developer
IBM
Bengaluru,
India
5-7 Years
2 out of 6 preferred skills are a match
Applied
new job description bg glownew job description bg glownew job description bg svg
Posted 8 hours ago
Be among the first 10 applicants
Early Applicant
Job Description
Introduction

At IBM Infrastructure & Technology, we design and operate the systems that keep the world running. From high-resiliency mainframes and hybrid cloud platforms to networking, automation, and site reliability. Our teams ensure the performance, security, and scalability that clients and industries depend on every day. Working in Infrastructure & Technology means tackling complex challenges with curiosity and collaboration. You'll work with diverse technologies and colleagues worldwide to deliver resilient, future-ready solutions that power innovation. With continuous learning, career growth, and a supportive culture, IBM provides the opportunities to build expertise and shape the infrastructure that drives progress.

Your Role And Responsibilities

As an AI Back-End Developer specializing in Artificial Intelligence for IBM Z, you will work in an agile, collaborative environment to design, code, and deliver opensource solutions to AI frameworks. You will employ IBM's Design Thinking to create solutions that provide a great user experience along with world-class quality, resiliency, performance, security, and stability. Your primary responsibilities will include:

Design and Develop Solutions: Design, code, debug, test, and deliver creative solutions to problems and requirements in AI frameworks.
Collaborate with Teams: Work in an agile, collaborative environment to understand requirements and deliver high-quality solutions that meet client needs.
Apply Design Thinking: Employ IBM's Design Thinking to create solutions that provide a great user experience along with world-class quality, resiliency, performance, security, and stability.
Work on leading-edge projects using technologies like machine learning, deep learning, and other AI-related areas to drive innovation and solution delivery.
Ensure Solution Quality: Ensure that solutions meet the required standards for quality, resiliency, performance, security, and stability.

Required Technical And Professional Expertise

AI Development Experience: Experience with designing, coding, debugging, testing PyTorch Core & Internals.
Contribution to and extend PyTorch internals and backend architecture, including:
TorchInductor
TorchDynamo
AOTAutograd
Debug low-level PyTorch runtime issues (autograd, dispatcher, kernels, graph breaks).
Deep hands-on experience with PyTorch FSDP
Optimize model execution paths for training and inference.
Open-source contributions to PyTorch or related projects
Experience with proprietary or custom AI accelerators
Experience integrating AI frameworks with custom or proprietary accelerators
Advanced skills in C++ and Python
Experience with GPU programming or accelerator SDKs
Advanced Python skills and strong debugging ability
DevOps Methodologies: Experience working with DevOps methodologies to deliver high-quality solutions on AI frameworks ensuring world-class quality, resiliency, performance, security, and stability.

Understanding Of

Preferred technical and professional experience

AI runtimes
Memory hierarchies
Parallel execution models
PyTorch distributed runtime
Parameter sharding and memory management
Hands-on experience with torch.compile and TorchInductor
Design Thinking Application: Experience employing IBM's Design Thinking to create solutions that provide a great user experience, with a focus on delivering innovative and effective solutions.
Experience contributing to leading-edge projects, driving innovation and solution delivery in Core backend AI areas.
Excellent communication skills
More Info
Job Type:
Permanent Job
Industry:
Other
Function:
Artificial Intelligence
Employment Type:
Full time
Key Skills
2 out of 6 preferred skills are a match

GPU programming


DevOps methodologies


Deep Learning

Machine Learning
Remove
Pytorch

Python
Remove
You can add or edit skills in profile page

About Company
IBM
IBM
Job ID: 146720095

Report Job
Recommended Jobs for you
View All
Jobs by Skill - IT




















Jobs by Skill - Non IT




















International Jobs




















Last Updated: 02-05-2026 05:22:57 AM
Home
jobs in Bengaluru / Bangalore
AI back-end Developer
Similar Jobs
View All
AI Performance Analyst
IBM
I
5-10 yrs
Bengaluru, India
Save
Quick Apply
Posted 16 hours ago
IND Senior Software Engineer
Walmart
W
1-6 yrs
Bengaluru
Save
Quick Apply
Posted 2 days ago
AI/ML Engineer
Saral Infotech Solution
S
3-5 yrs
₹ 1 - 10 LPA
Hyderabad, Bengaluru, Pune
Save
Quick Apply
Posted 19 hours ago
Python Developer - Product Owner
Synechron Technologies Private Limited
Synechron Technologies Private Limited
7-12 yrs
Bengaluru
Save
Quick Apply
Posted 2 days ago
Senior Data Engineer, Data Factory
Thomson Reuters
Thomson Reuters
6-9 yrs
Bengaluru
Save
Quick Apply
Posted 2 days ago
View All
Beware of Scammers
Beware of Scammers
We don’t charge any money for job offers

Interview Calls
What it feels like to have

48% more interview calls?
To get 5X more recruiter views on your profile

Ask here
BB-Desktop-JD-Right
Job Categories

Employers

Job Seekers

Career Advice
Company Info

IT Jobs

Non IT Jobs

Selected Country
India
Toll No: +91 80 6985 7811 |Toll Free No: 1800-419-6666
info@foundit.in
Download The App
IOS Download
Android Download
Stay Connected
Security & Fraud|Privacy Notice|Terms of Use|Beware of Fraudsters|Be SafeComplaints|
© 2026 foundit | All rights Reserved
'''         # raw JD text (str)


In [ ]:
import re
import csv
import json
from pathlib import Path
import pdfplumber
from difflib import SequenceMatcher
import spacy
from spacy.matcher import PhraseMatcher

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Installing spaCy model...")
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")


In [ ]:
RESUME_ANCHORS = {
    "summary": ["professional summary", "summary", "about me", "career objective", "profile", "executive summary", "professional profile", "overview"],
    "skills": ["skills", "technical skills", "core competencies", "technologies", "technical competencies", "tech stack", "tools", "toolset", "technical proficiencies"],
    "experience": ["professional experience", "work experience", "experience", "employment history", "career history", "work history", "professional background", "employment"],
    "projects": ["technical projects", "projects", "personal projects", "key projects", "project experience"],
    "education": ["education", "academic background", "qualifications", "academic credentials", "education and certifications"],
    "certifications": ["certifications", "certificates", "courses", "training", "professional development", "licenses"],
}

JD_ANCHORS = {
    "responsibilities": ["key responsibilities", "responsibilities", "what you will do", "duties", "main responsibilities"],
    "prerequisites": ["prerequisites", "requirements", "minimum requirements", "basic requirements"],
    "must_have": ["must have", "required skills", "essential skills", "mandatory", "required", "core requirements"],
    "nice_to_have": ["nice to have", "preferred", "good to have", "bonus", "preferred skills", "desired skills"],
    "optional": ["optional", "additional", "desired"],
}


In [ ]:

def fuzzy_score(a: str, b: str) -> float:
    return SequenceMatcher(None, a.lower().strip(), b.lower().strip()).ratio()

def fix_pdf_line_breaks(text: str) -> str:
    lines = text.split('\n')
    fixed = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            fixed.append('')
            continue
        is_continuation = (
            stripped[0].islower() or
            (not stripped.startswith('•') and
             not stripped.startswith('-') and
             not stripped[0].isupper() and
             len(stripped) > 5)
        )
        if is_continuation and fixed and fixed[-1]:
            fixed[-1] += ' ' + stripped
        else:
            fixed.append(stripped)
    return '\n'.join(fixed)

def detect_section(line: str, anchors: dict, threshold: float = 0.60):
    clean = re.sub(r'[^a-zA-Z\s&\-]', '', line).strip().lower()
    if len(clean) < 3 or len(clean) > 60:
        return None

    best_section, best_score = None, 0.0
    line_words = set(clean.split())
    
    for section, anchor_list in anchors.items():
        for anchor in anchor_list:
            anchor_words = set(anchor.split())
            if not anchor_words: continue
            exact_matches = line_words & anchor_words
            if exact_matches:
                score = 0.95 + (len(exact_matches) / len(anchor_words)) * 0.05
                if score > best_score:
                    best_score = score
                    best_section = section
    
    if best_score < 0.85:
        best_score = 0.0
        for section, anchor_list in anchors.items():
            for anchor in anchor_list:
                score = fuzzy_score(clean, anchor)
                if score > best_score:
                    best_score = score
                    best_section = section

    return best_section if best_score >= threshold else None

def split_text_by_sections(text: str, anchors: dict) -> dict:
    lines = text.split('\n')
    headers = []
    for i, line in enumerate(lines):
        clean_line = line.strip()
        if not clean_line: continue
        section = detect_section(clean_line, anchors)
        if section:
            if headers and headers[-1][1] == section and i - headers[-1][0] <= 2:
                continue
            headers.append((i, section))
    sections = {}
    for idx, (line_no, name) in enumerate(headers):
        end = headers[idx + 1][0] if idx + 1 < len(headers) else len(lines)
        block = '\n'.join(lines[line_no + 1:end]).strip()
        if not block: continue
        if name not in sections or len(block) > len(sections[name]):
            sections[name] = block
    return sections


In [ ]:

def extract_contact(text: str) -> dict:
    contact = {"name": "", "email": "", "phone": "", "linkedin": "", "github": ""}
    email = re.search(r'[\w.+-]+@[\w-]+\.[\w.]+', text)
    if email: contact["email"] = email.group()

    phone = re.search(r'(\+?\d[\d\-\s]{8,15}\d)', text)
    if phone: contact["phone"] = phone.group()
    
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if lines:
        first_line = lines[0]
        match = re.match(r'^([^@\d\|]+?)(?:[\d\|]|[\w.+-]+@|$)', first_line)
        if match:
            potential_name = match.group(1).strip()
            potential_name = re.sub(r'[|•\-]+$', '', potential_name).strip()
            if potential_name and 2 < len(potential_name) < 80:
                if not any(kw in potential_name.lower() for kw in ['professional', 'summary', 'objective', 'experience', 'education', 'skill']):
                    contact["name"] = potential_name
        
        if not contact["name"]:
            for line in lines[1:15]:
                if ':' in line or len(line) > 80: continue
                words = line.split()
                if 1 <= len(words) <= 5 and line[0].isupper():
                    if not any(char in line for char in ['@', ':', '•']) and not any(kw in line.lower() for kw in ['professional', 'summary', 'objective', 'experience', 'education', 'skill']):
                        contact["name"] = line
                        break
    
    if 'linkedin' in text.lower(): contact["linkedin"] = "Yes"
    if 'github' in text.lower(): contact["github"] = "Yes"
    return contact

def parse_experience(text: str) -> list:
    if not text or not text.strip(): return []

    year_pattern = r"(?:19|20)\d{2}"
    month_year = rf"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)[\s\-]*{year_pattern}"
    date_pattern = rf"(\b{month_year}\b|\b{year_pattern}\b|Present|Current|Till Date)"
    
    title_keywords = r"(engineer|developer|manager|director|analyst|consultant|lead|senior|jr|intern|architect|scientist|specialist|officer|associate|programmer|researcher|assistant)"

    normalized = text.replace("\uf0b7", "\n• ")
    normalized = re.sub(r"\s+[•*\-]\s+", "\n• ", normalized)
    lines = [line.strip() for line in normalized.split("\n") if line.strip()]
    if not lines: return []

    def strip_bullet(line: str) -> str: return re.sub(r"^[•\-\*\s]+", "", line).strip()

    def is_header_candidate(line: str) -> bool:
        clean = strip_bullet(line)
        if len(clean) < 6: return False
        if clean.startswith("•") or line.startswith(("•", "-", "*")): return False
        has_title = bool(re.search(title_keywords, clean, flags=re.IGNORECASE))
        has_date = bool(re.search(date_pattern, clean, flags=re.IGNORECASE))
        doc = nlp(clean)
        has_org = any(ent.label_ in ("ORG", "GPE") for ent in doc.ents)
        has_comma = "," in clean or "|" in clean or "-" in clean
        if has_title and (has_date or has_org or has_comma): return True
        if has_date and re.search(r"(engineer|developer|scientist|analyst|intern|researcher)$", clean.replace(" ", "").lower()): return True
        return False

    groups = []
    current = []
    seen_header = False

    for line in lines:
        if is_header_candidate(line):
            if current and seen_header:
                groups.append(current)
                current = []
            seen_header = True
            current.append(line)
            continue
        if seen_header: current.append(line)

    if current and seen_header: groups.append(current)
    if not groups: groups = [lines]

    entries = []
    for group in groups:
        header = strip_bullet(group[0])

        all_dates = re.findall(date_pattern, " ".join(group), flags=re.IGNORECASE)
        from_date = all_dates[0].strip() if len(all_dates) >= 1 else ""
        to_date = all_dates[1].strip() if len(all_dates) >= 2 else ""

        header_no_date = re.sub(date_pattern, "", header, flags=re.IGNORECASE)
        header_no_date = re.sub(r"\s+", " ", header_no_date).strip(" ,.-–—|")

        parts = [p.strip() for p in re.split(r"[,|\-]", header_no_date) if p.strip()]
        designation = ""
        company = ""
        
        if len(parts) >= 2:
            for part in parts:
                if re.search(title_keywords, part, re.IGNORECASE): designation = part
                else:
                    doc_part = nlp(part)
                    if any(ent.label_ in ("ORG", "GPE") for ent in doc_part.ents): company = part
            if not designation: designation = parts[0]
            if not company: company = parts[1]
        else:
            designation = parts[0] if parts else header_no_date
            doc_header = nlp(header_no_date)
            orgs = [ent.text for ent in doc_header.ents if ent.label_ in ("ORG", "GPE")]
            if orgs: company = orgs[0]

        responsibilities = []
        for line in group[1:]:
            if line.startswith(("•", "-", "*")):
                item = strip_bullet(line)
                if item: responsibilities.append(item)

        if not responsibilities:
            sentence_candidates = re.split(r"(?<=[.!?])\s+", " ".join(group[1:]))
            for sentence in sentence_candidates:
                sentence = sentence.strip()
                doc_sent = nlp(sentence)
                if len(doc_sent) > 2 and doc_sent[0].pos_ == "VERB": responsibilities.append(sentence)
                elif re.match(r"^(Led|Built|Implemented|Managed|Developed|Engineered|Designed|Worked|Created|Automated|Improved|Performed|Conducted|Collaborated|Maintained|Optimized|Architected|Integrated|Developing)\b", sentence, flags=re.IGNORECASE): responsibilities.append(sentence)

        entries.append({"company": company, "designation": designation, "from": from_date, "to": to_date, "responsibilities": responsibilities})

    return entries

def parse_skills_section(text: str) -> dict:
    if not text.strip(): return {}
    skills_dict = {"Technical Skills": []}
    lines = text.split("\n")
    current_category = "Technical Skills"
    for line in lines:
        line = line.strip()
        if not line: continue
        if ":" in line and len(line.split(":")[0]) < 30:
            parts = line.split(":")
            current_category = parts[0].strip()
            skills_dict[current_category] = []
            skills_str = parts[1].strip()
            if skills_str:
                skills_dict[current_category].extend([s.strip() for s in re.split(r"[,|•]", skills_str) if s.strip()])
        else:
            if current_category not in skills_dict: skills_dict[current_category] = []
            skills_dict[current_category].extend([s.strip() for s in re.split(r"[,|•]", line) if s.strip()])
            
    for cat in list(skills_dict.keys()):
        cleaned = []
        for s in skills_dict[cat]:
            if len(s) > 1 and s.lower() not in [x.lower() for x in cleaned]: cleaned.append(s)
        skills_dict[cat] = cleaned
        if not cleaned: del skills_dict[cat]
            
    return skills_dict

def parse_education(text: str) -> list:
    if not text.strip(): return []
    year_pattern = r"(?:19|20)\d{2}"
    month_year = rf"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)[\s\-]*{year_pattern}"
    date_pattern = rf"(\b{month_year}\b|\b{year_pattern}\b|Present|Current|Till Date)"
    degree_keywords = r"(Bachelor|Master|Ph\.?D|B\.?S|M\.?S|B\.?A|M\.?A|B\.?E|M\.?E|B\.?Tech|M\.?Tech|BSc|MSc|Degree|Diploma|Secondary|CBSE|ICSE)"
    
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    entries = []
    current_entry = {}
    
    for line in lines:
        if not line: continue
        doc = nlp(line)
        dates = re.findall(date_pattern, line, re.IGNORECASE)
        if dates and "from" not in current_entry:
            current_entry["from"] = dates[0]
            if len(dates) > 1: current_entry["to"] = dates[1]
                
        orgs = [ent.text for ent in doc.ents if ent.label_ in ("ORG", "GPE")]
        college = orgs[0] if orgs else ""
        if ("university" in line.lower() or "college" in line.lower() or "institute" in line.lower() or "school" in line.lower()):
            if not college: college = line
        
        degree_match = re.search(degree_keywords, line, re.IGNORECASE)
        
        if college and "college" not in current_entry: current_entry["college"] = college
        if degree_match and "degree" not in current_entry: current_entry["degree"] = line
        elif not degree_match and not college and len(line) > 5 and "specializations" not in current_entry: current_entry["specializations"] = line
            
        if "college" in current_entry and "degree" in current_entry and len(current_entry) >= 2:
            if degree_match and current_entry.get("degree") != line:
                entries.append(current_entry)
                current_entry = {"degree": line}
                
    if current_entry: entries.append(current_entry)
    return entries


In [ ]:

def parse_resume_pdf(pdf_path: str) -> dict:
    def normalize_pdf_text(raw_text: str) -> str:
        text = raw_text or ""
        replacements = {"\ufb00": "ff", "\ufb01": "fi", "\ufb02": "fl", "\uf0b7": "•", "\u2013": "-", "\u2014": "-"}
        for old, new in replacements.items(): text = text.replace(old, new)
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    def extract_text_pymupdf(path: str) -> str:
        if fitz is None: return ""
        pages_text = []
        doc = fitz.open(path)
        try:
            for page in doc:
                blocks = page.get_text("blocks")
                blocks = sorted(blocks, key=lambda b: (round(b[1], 1), round(b[0], 1)))
                block_lines = []
                for block in blocks:
                    block_text = (block[4] or "").strip()
                    if block_text: block_lines.append(block_text)
                page_text = "\n".join(block_lines).strip()
                if page_text: pages_text.append(page_text)
        finally: doc.close()
        return "\n\n".join(pages_text).strip()

    raw_text = extract_text_pymupdf(pdf_path)
    if not raw_text:
        with pdfplumber.open(pdf_path) as pdf:
            raw_text = "\n".join(page.extract_text() or "" for page in pdf.pages)

    text = normalize_pdf_text(raw_text)
    text = fix_pdf_line_breaks(text)
    sections = split_text_by_sections(text, RESUME_ANCHORS)

    def capture_block_by_keywords(source_text: str, start_keywords: list, stop_section_keys: set) -> str:
        lines = source_text.split('\n')
        captured = []
        in_block = False
        for line in lines:
            raw = line.strip()
            if not raw:
                if in_block and captured: captured.append("")
                continue
            lowered = raw.lower()
            if not in_block and any(keyword in lowered for keyword in start_keywords):
                in_block = True
                continue
            if in_block:
                detected = detect_section(raw, RESUME_ANCHORS)
                if detected and detected in stop_section_keys: break
                captured.append(raw)
        return "\n".join(captured).strip()

    summary_text = sections.get("summary", "")
    skills_text = sections.get("skills", "")
    experience_text = sections.get("experience", "")
    education_text = sections.get("education", "")

    if not summary_text: summary_text = capture_block_by_keywords(text, RESUME_ANCHORS.get("summary", []), {"experience", "projects", "education", "skills", "certifications"})
    if not skills_text: skills_text = capture_block_by_keywords(text, RESUME_ANCHORS.get("skills", []), {"experience", "projects", "education", "certifications", "summary"})
    if not experience_text: experience_text = capture_block_by_keywords(text, RESUME_ANCHORS.get("experience", []), {"projects", "education", "certifications", "skills", "summary"})
    if not education_text: education_text = capture_block_by_keywords(text, RESUME_ANCHORS.get("education", []), {"experience", "projects", "skills", "certifications", "summary"})

    return {
        "contact": extract_contact(text),
        "summary": summary_text,
        "skills": parse_skills_section(skills_text),
        "experience": parse_experience(experience_text),
        "education": parse_education(education_text),
    }

def load_skill_terms(paths: list) -> list:
    terms = []
    candidate_columns = {"skill", "skills", "name", "label", "preferredlabel", "preferred_label", "technology", "technologies", "competency", "competencies", "elementname", "title", "occupation", "description"}
    def normalize_key(value: str) -> str: return str(value).lower().replace(" ", "").replace("_", "")
    for raw_path in paths:
        path = Path(raw_path)
        if not path.exists() or not path.is_file(): continue
        try:
            suffix = path.suffix.lower()
            if suffix == ".txt":
                with path.open("r", encoding="utf-8") as file: terms.extend([line.strip() for line in file if line.strip()])
            elif suffix == ".csv":
                with path.open("r", encoding="utf-8", newline="") as file:
                    reader = csv.DictReader(file)
                    field_map = {normalize_key(field): field for field in (reader.fieldnames or [])}
                    matched_cols = [field_map[key] for key in field_map if key in candidate_columns]
                    if matched_cols:
                        for row in reader:
                            for column in matched_cols:
                                value = (row.get(column) or "").strip()
                                if value: terms.append(value)
                    else:
                        file.seek(0)
                        fallback_reader = csv.reader(file)
                        for row in fallback_reader:
                            for value in row:
                                value = str(value).strip()
                                if value and len(value) > 1: terms.append(value)
        except Exception as error: print(f"Skipped dataset {path}: {error}")
    cleaned = []
    seen = set()
    for term in terms:
        normalized = re.sub(r"\s+", " ", str(term)).strip()
        key = normalized.lower()
        if len(normalized) > 1 and key not in seen:
            cleaned.append(normalized)
            seen.add(key)
    return cleaned

def build_skill_matcher(skill_terms: list):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    if skill_terms:
        patterns = [nlp.make_doc(term) for term in skill_terms]
        matcher.add("SKILL", patterns)
    return matcher

try:
    SKILL_TERMS = load_skill_terms(["../data/esco_skills.csv", "../data/onet_skills.xlsx", "../data/skills.txt"])
    SKILL_MATCHER = build_skill_matcher(SKILL_TERMS)
    print(f"Loaded skill terms: {len(SKILL_TERMS)}")
except Exception as e:
    SKILL_TERMS = []
    SKILL_MATCHER = None

def extract_skills_spacy(sentence: str) -> list:
    if not sentence or len(sentence.strip()) < 3: return []
    doc = nlp(sentence)
    skills = []
    if SKILL_MATCHER:
        for _, start, end in SKILL_MATCHER(doc):
            span = doc[start:end]
            value = span.text.strip()
            if value: skills.append(value)
    
    # NLP fallback without manual skip list
    for chunk in doc.noun_chunks:
        tokens = [token for token in chunk if not token.is_space]
        if not tokens: continue
        content_tokens = [token for token in tokens if not token.is_stop and not token.is_punct and not token.like_num and len(token.text) > 1]
        if not content_tokens: continue
        if len(content_tokens) <= 4 and any(token.pos_ in {"NOUN", "PROPN", "ADJ"} for token in content_tokens):
            candidate = " ".join(token.text for token in content_tokens).strip()
            if candidate: skills.append(candidate)
            
    for token in doc:
        if token.is_stop or token.is_punct or token.like_num or len(token.text) < 2: continue
        if token.pos_ in {"NOUN", "PROPN", "X"}: skills.append(token.text)

    unique_skills = []
    seen = set()
    for skill in skills:
        cleaned = re.sub(r"\s+", " ", skill).strip(" .,:;()[]{}")
        key = cleaned.lower()
        if cleaned and len(cleaned) > 1 and key not in seen:
            unique_skills.append(cleaned)
            seen.add(key)
    return unique_skills

def parse_jd(jd_text: str) -> dict:
    sections = split_text_by_sections(jd_text, JD_ANCHORS)
    def items(text): return [re.sub(r'^[•\-\d.)\]]\s*', '', l).strip() for l in text.split('\n') if len(l.strip()) > 3]
    all_items = {}
    for key, value in sections.items(): all_items[key] = items(value) if value else []

    prerequisites = all_items.get("prerequisites", [])
    must_have_raw = all_items.get("must_have", [])
    nice_to_have_raw = all_items.get("nice_to_have", [])
    
    def filter_valid_skills(skill_list):
        valid = []
        for s in skill_list:
            doc = nlp(s)
            if len(doc) > 0 and doc[0].pos_ not in ("VERB", "ADV", "PRON", "DET"):
                if "description" not in s.lower() and "bg" not in s.lower() and "svg" not in s.lower() and "job" not in s.lower():
                    valid.append(s)
        return valid

    must_have_skills = []
    for sentence in must_have_raw: must_have_skills.extend(extract_skills_spacy(sentence))
    nice_to_have_skills = []
    for sentence in nice_to_have_raw: nice_to_have_skills.extend(extract_skills_spacy(sentence))

    must_have_skills = list(dict.fromkeys(filter_valid_skills(must_have_skills)))
    nice_to_have_skills = list(dict.fromkeys(filter_valid_skills(nice_to_have_skills)))

    must_have_lower = {skill.lower() for skill in must_have_skills}
    nice_to_have_skills = [skill for skill in nice_to_have_skills if skill.lower() not in must_have_lower]

    return {
        "responsibilities": all_items.get("responsibilities", []),
        "prerequisites": prerequisites,
        "must_have": must_have_skills,
        "nice_to_have": nice_to_have_skills,
    }

resume_data = parse_resume_pdf(resume_path)
jd_data = parse_jd(jd_texts)


In [ ]:
print("=" * 80)
print("✅ STRUCTURED RESUME DATA")
print("=" * 80)
print("\n🎓 EDUCATION:")
for i, edu in enumerate(resume_data['education'], 1):
    print(f"\n  [{i}]")
    print(f"      College: {edu.get('college', 'N/A')}")
    print(f"      Degree: {edu.get('degree', 'N/A')}")
    if edu.get('specializations'):
        print(f"      Specializations: {edu.get('specializations', 'N/A')}")
    print(f"      From: {edu.get('from', 'N/A')} → To: {edu.get('to', 'N/A')}")
print("\n💼 EXPERIENCE:")
for i, exp in enumerate(resume_data['experience'], 1):
    print(f"\n  [{i}]")
    print(f"      Company: {exp.get('company', 'N/A')}")
    print(f"      Designation: {exp.get('designation', 'N/A')}")
    print(f"      From: {exp.get('from', 'N/A')} → To: {exp.get('to', 'N/A')}")
    if exp.get('responsibilities'):
        print(f"      Responsibilities ({len(exp['responsibilities'])} items):")
        for resp in exp['responsibilities'][:2]:
            print(f"        ✓ {resp[:55]}...")
print("\n" + "=" * 80)
print("✅ All data structured correctly!")
print("=" * 80)

# Skills Extraction Summary
print("\n" + "="*80)
print("📊 SKILLS EXTRACTION SUMMARY")
print("="*80)
print("\n🔧 Resume Skills (by category):")
for category, skills_list in resume_data['skills'].items():
    print(f"\n  {category}:")
    for skill in skills_list[:5]:
        print(f"    • {skill}")
    if len(skills_list) > 5:
        print(f"    ... and {len(skills_list) - 5} more")

print("\n\n🎯 JD Must-Have Skills:")
for skill in jd_data['must_have'][:10]:
    print(f"  • {skill}")
if len(jd_data['must_have']) > 10:
    print(f"  ... and {len(jd_data['must_have']) - 10} more")

print("\n\n✨ JD Nice-to-Have Skills:")
for skill in jd_data['nice_to_have'][:10]:
    print(f"  • {skill}")
if len(jd_data['nice_to_have']) > 10:
    print(f"  ... and {len(jd_data['nice_to_have']) - 10} more")
